# Notebook 3: LLM pipelines and prompt chaining

**5eiai50 Artificial Intelligence project, 2026-2027. Session of 06/11.**

In notebook 2, our demonstration project was a single prompt that classifies a text by SDG. Today it becomes a **pipeline**:

```
text -> extract (structured) -> gate -> route by SDG family -> write a brief -> verify -> output
```

You will:

1. write versioned **prompt templates** with LangChain;
2. get **structured outputs** validated by Pydantic;
3. **chain** calls with gates and routing in plain Python;
4. run steps **in parallel** (sectioning and voting);
5. compare a **monolithic prompt** with the **pipeline**: accuracy, tokens and estimated footprint, **per step**;
6. (bonus) add an **evaluator-optimizer** loop.

Exercises are marked with ✏️. Estimated time: 60 minutes.

## 0. Setup

In [ ]:
%pip install -q langchain-core langchain-mistralai mistralai ecologits python-dotenv pandas

/home/valatras/Documents/vscode_projects/mistral-sdg-project/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("MISTRAL_API_KEY"), "MISTRAL_API_KEY not found: check your .env file"

from langchain_mistralai import ChatMistralAI

TINY = "ministral-3b-latest"
SMALL = "mistral-small-latest"
LARGE = "mistral-large-latest"

tiny = ChatMistralAI(model=TINY, temperature=0)
small = ChatMistralAI(model=SMALL, temperature=0)
large = ChatMistralAI(model=LARGE, temperature=0)

# Copy the prices you found in notebook 2 (EUR per million tokens: input, output).
PRICES = {TINY: None, SMALL: None, LARGE: None}

### `tracker.py`, version 2

LangChain calls the model for us, so we can no longer wrap `client.chat.complete`. Instead we use a **callback**: LangChain notifies it at the start and at the end of every model call, wherever it happens in the chain.

The callback also records the **tags** of the step (`step:extract`, `step:brief`...), so that we can measure the cost of each step separately. The new version keeps `tracked_chat` from notebook 2: replace the file in your project repository.

In [5]:
%%writefile tracker.py
"""Logs every LLM call: model, prompt version, step, tokens, latency, cost and estimated footprint."""
import csv
import time
from datetime import datetime
from pathlib import Path

from ecologits.tracers.utils import llm_impacts
from langchain_core.callbacks import BaseCallbackHandler

LOG_FILE = Path("calls_log.csv")
FIELDS = ["timestamp", "model", "prompt_version", "tag", "prompt_tokens", "completion_tokens",
          "latency_s", "cost_eur", "energy_Wh", "gwp_gCO2e", "water_mL"]


def _check_header():
    # If the log was written by an older version of this module (other columns), keep it aside.
    if LOG_FILE.exists():
        with LOG_FILE.open(encoding="utf-8") as f:
            header = f.readline().strip().split(",")
        if header != FIELDS:
            LOG_FILE.rename(LOG_FILE.with_name(f"calls_log_old_{datetime.now():%Y%m%d_%H%M%S}.csv"))


def _value(x):
    return (x.min + x.max) / 2 if hasattr(x, "min") else x


def log_call(model, prompt_tokens, completion_tokens, latency, prices=None, prompt_version="v0", tag=""):
    impacts = llm_impacts(provider="mistralai", model_name=model,
                          output_token_count=completion_tokens, request_latency=latency)
    price = (prices or {}).get(model)
    cost = (prompt_tokens * price[0] + completion_tokens * price[1]) / 1e6 if price else None
    row = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": model, "prompt_version": prompt_version, "tag": tag,
        "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
        "latency_s": round(latency, 3), "cost_eur": cost,
        "energy_Wh": _value(impacts.energy.value) * 1000,
        "gwp_gCO2e": _value(impacts.gwp.value) * 1000,
        "water_mL": _value(impacts.wcf.value) * 1000,
    }
    _check_header()
    new_file = not LOG_FILE.exists()
    with LOG_FILE.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)


def tracked_chat(client, model, messages, prices=None, prompt_version="v0", tag="", **kwargs):
    """Mistral SDK: calls client.chat.complete and logs the call (notebook 2)."""
    start = time.perf_counter()
    response = client.chat.complete(model=model, messages=messages, **kwargs)
    usage = response.usage
    log_call(model, usage.prompt_tokens, usage.completion_tokens, time.perf_counter() - start,
             prices, prompt_version, tag)
    return response


class TokenTracker(BaseCallbackHandler):
    """LangChain: logs every chat model call of a chain. The step is read from tags 'step:<name>'."""

    def __init__(self, prices=None, prompt_version="v0"):
        self.prices, self.prompt_version = prices, prompt_version
        self._starts = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, tags=None, **kwargs):
        model = (kwargs.get("invocation_params") or {}).get("model", "unknown")
        steps = [t[5:] for t in (tags or []) if t.startswith("step:")]
        self._starts[run_id] = (time.perf_counter(), model, steps[-1] if steps else "")

    def on_llm_end(self, response, *, run_id, **kwargs):
        start, model, step = self._starts.pop(run_id, (time.perf_counter(), "unknown", ""))
        output = response.llm_output or {}
        model = output.get("model_name", model)
        usage = output.get("token_usage") or {}
        prompt_tokens, completion_tokens = usage.get("prompt_tokens"), usage.get("completion_tokens")
        if prompt_tokens is None:  # fall back on the message metadata
            meta = getattr(response.generations[0][0].message, "usage_metadata", None) or {}
            prompt_tokens, completion_tokens = meta.get("input_tokens", 0), meta.get("output_tokens", 0)
        log_call(model, prompt_tokens, completion_tokens, time.perf_counter() - start,
                 self.prices, self.prompt_version, step)


def load_log():
    import pandas as pd
    log = pd.read_csv(LOG_FILE)
    # Logs written by an older version of this module had the cost in USD.
    return log.rename(columns={"cost_usd": "cost_eur"})


Overwriting tracker.py


In [6]:
import importlib
import tracker
importlib.reload(tracker)
assert "cost_eur" in getattr(tracker, "FIELDS", []), (
    "tracker.py is an old version: run the %%writefile tracker.py cell of this notebook (or of notebook 3)")


def run(chain, inputs, prompt_version, batch=False):
    # Runs a chain (on one input or a list) with a tracker attached; every model call is logged.
    config = {"callbacks": [tracker.TokenTracker(PRICES, prompt_version)], "max_concurrency": 4}
    return chain.batch(inputs, config=config) if batch else chain.invoke(inputs, config=config)

The same ten examples as in notebook 2 (expected SDG, `None` when no SDG clearly applies).

In [7]:
EXAMPLES = [
    ("The municipality installed rainwater tanks in 40 schools to reduce tap water use during droughts.", 6),
    ("A cooperative trains women farmers to access microcredit and own the land they cultivate.", 5),
    ("The new tram line connects the suburbs to the city centre and reduces car traffic.", 11),
    ("Free school meals were extended to all primary pupils in the district.", 2),
    ("The hospital launched a mobile clinic to vaccinate children in remote villages.", 3),
    ("Solar panels and batteries now power the health centres of three islands.", 7),
    ("Volunteers removed two tonnes of plastic from the beach and the coral reef.", 14),
    ("A reforestation programme planted 50,000 native trees on degraded hillsides.", 15),
    ("The football club won the regional championship after a dramatic final.", None),
    ("An online platform lets citizens track how the municipal budget is spent.", 16),
]
TEXTS = [t for t, _ in EXAMPLES]

## 1. Prompt templates

A template separates the fixed instructions (written by you) from the variable data (often written by others). We keep all prompts in one dictionary with a **version id**: the id is logged with every call.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

PROMPTS = {
    "extract-v1": ChatPromptTemplate.from_messages([
        ("system",
         "You analyse short local news for an NGO that tracks the UN Sustainable Development Goals (SDGs 1-17).\n"
         "Extract the requested fields from the text only. Do not infer facts that are not stated.\n"
         "If no SDG clearly applies, set sdg to null."),
        ("human", "<text>\n{text}\n</text>"),
    ]),
}

messages = PROMPTS["extract-v1"].invoke({"text": TEXTS[0]})
for m in messages.to_messages():
    print(f"[{m.type}]\n{m.content}\n")

[system]
You analyse short local news for an NGO that tracks the UN Sustainable Development Goals (SDGs 1-17).
Extract the requested fields from the text only. Do not infer facts that are not stated.
If no SDG clearly applies, set sdg to null.

[human]
<text>
The municipality installed rainwater tanks in 40 schools to reduce tap water use during droughts.
</text>



In [ ]:
# ✏️ Exercise 1 (solution)
try:
    PROMPTS["extract-v1"].invoke({"text": TEXTS[0]})
except KeyError as e:
    print("Missing variable detected before any call:", e)

with_example = ChatPromptTemplate.from_messages([
    ("system", 'Answer in JSON, for example {{"sdg": 6}}.'),   # doubled braces = literal braces
    ("human", "{text}"),
])
print(with_example.invoke({"text": "..."}).to_messages()[0].content)

Missing variable detected before any call: "Input to ChatPromptTemplate is missing variables {'text'}.  Expected: ['text'] Received: []\nNote: if you intended {text} to be part of the string and not a variable, please escape it with double curly braces like: '{{text}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "
Answer in JSON, for example {"sdg": 6}.


## 2. Structured outputs

The output of the extraction step is consumed by **code** (the gate and the router), so it must follow a schema. We describe it with Pydantic: the field descriptions are sent to the model, and Pydantic validates the values.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError


class SDGAnalysis(BaseModel):
    actors: list[str] = Field(description="Organisations or groups that act, as named in the text")
    place: Optional[str] = Field(default=None, description="Where it happens, if stated")
    action: str = Field(description="What is done, in at most 15 words")
    sdg: Optional[int] = Field(default=None, ge=1, le=17,
                               description="The ONE SDG the text contributes to most directly, null if none")
    confidence: Literal["low", "medium", "high"] = Field(description="Your confidence in the SDG")


extract_chain = (PROMPTS["extract-v1"] | tiny.with_structured_output(SDGAnalysis)).with_config(tags=["step:extract"])

analysis = run(extract_chain, {"text": TEXTS[0]}, prompt_version="extract-v1")
analysis

Even when the API respects the schema, the **values** must be checked. Pydantic does it for us:

In [ ]:
try:
    SDGAnalysis(actors=["city"], action="tanks", sdg=18, confidence="high")
except ValidationError as e:
    print(e)

In [ ]:
# ✏️ Exercise 2 (solution)
from pydantic import model_validator


class SDGAnalysisWithTarget(SDGAnalysis):
    target: Optional[str] = Field(default=None, pattern=r"^\d{1,2}\.[0-9a-z]$",
                                  description="The SDG target, e.g. '6.4', null if unsure")

    @model_validator(mode="after")
    def target_matches_sdg(self):
        if self.target and self.sdg and not self.target.startswith(f"{self.sdg}."):
            raise ValueError(f"target {self.target} does not belong to SDG {self.sdg}")
        return self


extract_target_chain = (PROMPTS["extract-v1"] | small.with_structured_output(SDGAnalysisWithTarget)
                        ).with_config(tags=["step:extract"])
for text in TEXTS[:3]:
    print(run(extract_target_chain, {"text": text}, prompt_version="extract-target-v1"))

## 3. Chaining: gate, routing, brief, verification

The pipeline keeps a **state dictionary** that grows step by step. Gates and routing are **plain Python**: we can test them without calling any model.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

FAMILY_OF = {**{n: "People" for n in (1, 2, 3, 4, 5)},
             **{n: "Prosperity" for n in (7, 8, 9, 10, 11)},
             **{n: "Planet" for n in (6, 12, 13, 14, 15)},
             **{n: "Peace and partnership" for n in (16, 17)}}

FOCUS = {
    "People": "the people who benefit, and how their living conditions change",
    "Prosperity": "the economic and infrastructure aspects, and who can access them",
    "Planet": "the environmental resource or ecosystem concerned, and the order of magnitude of the effect",
    "Peace and partnership": "the institutions involved, transparency and participation",
}

PROMPTS["brief-v1"] = ChatPromptTemplate.from_messages([
    ("system",
     "You write briefs for an NGO dashboard about SDG {sdg}.\n"
     "Focus on {focus}.\n"
     "Write at most 80 words, then one line 'Indicator to follow: ...' with a measurable indicator.\n"
     "Use only facts stated in the text."),
    ("human", "<text>\n{text}\n</text>"),
])


class Verdict(BaseModel):
    grounded: bool = Field(description="True if every fact of the brief is stated in the text")
    unsupported_claims: list[str] = Field(default_factory=list, description="Facts of the brief absent from the text")


PROMPTS["verify-v1"] = ChatPromptTemplate.from_messages([
    ("system", "You check a brief against its source text. List every fact of the brief that is not stated in the text."),
    ("human", "<text>\n{text}\n</text>\n<brief>\n{brief}\n</brief>"),
])


def brief_chain_for(family):
    return (RunnablePassthrough.assign(sdg=lambda s: s["analysis"].sdg, focus=lambda s: FOCUS[family])
            | PROMPTS["brief-v1"] | small | StrOutputParser()).with_config(tags=["step:brief"])


BRIEF_CHAINS = {family: brief_chain_for(family) for family in FOCUS}
verify_chain = (PROMPTS["verify-v1"] | tiny.with_structured_output(Verdict)).with_config(tags=["step:verify"])


def route(state):
    analysis = state["analysis"]
    if analysis.sdg is None:                                   # gate: nothing to write
        return {**state, "stopped": "no SDG identified"}
    family = FAMILY_OF[analysis.sdg]                           # routing in plain code
    return (RunnablePassthrough.assign(brief=BRIEF_CHAINS[family])
            | RunnablePassthrough.assign(verdict=verify_chain))


pipeline = RunnablePassthrough.assign(analysis=extract_chain) | RunnableLambda(route)

The gate and the router can be tested **without any model call**:

In [ ]:
no_sdg = SDGAnalysis(actors=["club"], action="won a match", sdg=None, confidence="high")
assert route({"text": "...", "analysis": no_sdg})["stopped"] == "no SDG identified"

water = SDGAnalysis(actors=["city"], action="rainwater tanks", sdg=6, confidence="high")
assert not isinstance(route({"text": "...", "analysis": water}), dict)   # returns a chain, not a result
print("gate and router tests passed")

In [ ]:
for text in [TEXTS[0], TEXTS[8]]:
    result = run(pipeline, {"text": text}, prompt_version="pipeline-v1")
    print("TEXT:", text)
    print("ANALYSIS:", result["analysis"])
    print(result.get("stopped") or f"BRIEF:\n{result['brief']}\nVERDICT: {result['verdict']}")
    print("-" * 80)

In [ ]:
log = tracker.load_log()
(log[log.prompt_version == "pipeline-v1"]
 .groupby(["tag", "model"])[["prompt_tokens", "completion_tokens", "gwp_gCO2e"]].sum())

In [ ]:
# ✏️ Exercise 3 (solution)
def route_v2(state):
    analysis = state["analysis"]
    if analysis.sdg is None:
        return {**state, "stopped": "no SDG identified"}
    if analysis.confidence == "low":                          # second gate: escalate to a human
        return {**state, "stopped": "low confidence: send to a human"}
    return route(state)


unsure = SDGAnalysis(actors=["?"], action="unclear", sdg=9, confidence="low")
assert route_v2({"text": "...", "analysis": unsure})["stopped"].startswith("low confidence")
pipeline_v2 = RunnablePassthrough.assign(analysis=extract_chain) | RunnableLambda(route_v2)
print("second gate tested")
# b) Usually the brief: it has the longest output (output tokens drive the estimated footprint)
#    and runs on the larger model. The verification re-reads the text and the brief (input tokens).

## 4. Parallelisation

**Sectioning**: independent sub-tasks run at the same time. **Voting**: the same task several times, the majority wins.

In [ ]:
import time
from langchain_core.runnables import RunnableParallel


def simple_chain(instruction, tag):
    prompt = ChatPromptTemplate.from_messages([("system", instruction), ("human", "<text>\n{text}\n</text>")])
    return (prompt | tiny | StrOutputParser()).with_config(tags=[f"step:{tag}"])


summary = simple_chain("Summarise the text in one sentence.", "summary")
risks = simple_chain("Name one possible negative side effect of the action described. One sentence.", "risks")
indicator = simple_chain("Propose one measurable indicator to follow the effect of this action. One line.", "indicator")

sections = RunnableParallel(summary=summary, risks=risks, indicator=indicator)

start = time.perf_counter()
parallel_result = run(sections, {"text": TEXTS[2]}, prompt_version="sections-parallel")
t_parallel = time.perf_counter() - start

start = time.perf_counter()
for name, chain in [("summary", summary), ("risks", risks), ("indicator", indicator)]:
    run(chain, {"text": TEXTS[2]}, prompt_version="sections-sequential")
t_sequential = time.perf_counter() - start

print(parallel_result)
print(f"parallel: {t_parallel:.1f} s, sequential: {t_sequential:.1f} s")

Voting: three different prompts classify the same text; the most frequent answer wins.

In [ ]:
from collections import Counter


class SDGOnly(BaseModel):
    sdg: Optional[int] = Field(default=None, ge=1, le=17, description="Main SDG, null if none")


VOTER_INSTRUCTIONS = [
    "Return the ONE SDG (1-17) the text contributes to most directly, or null.",
    "You are a UN analyst. Which single Sustainable Development Goal does this action serve best? Null if none.",
    "Think about who benefits and what changes, then return the main SDG number, or null if no SDG applies.",
]
voters = RunnableParallel({
    f"v{i}": (ChatPromptTemplate.from_messages([("system", instr), ("human", "<text>\n{text}\n</text>")])
              | tiny.with_structured_output(SDGOnly)).with_config(tags=["step:vote"])
    for i, instr in enumerate(VOTER_INSTRUCTIONS)
})


def majority(votes):
    counts = Counter(v.sdg for v in votes.values())
    return counts.most_common(1)[0][0]


vote_chain = voters | RunnableLambda(majority)
print(run(vote_chain, {"text": TEXTS[1]}, prompt_version="vote-v1"))

In [ ]:
# ✏️ Exercise 4 (solution)
single = (ChatPromptTemplate.from_messages([("system", VOTER_INSTRUCTIONS[0]), ("human", "<text>\n{text}\n</text>")])
          | tiny.with_structured_output(SDGOnly)).with_config(tags=["step:vote"])
inputs = [{"text": t} for t in TEXTS]
single_preds = [r.sdg for r in run(single, inputs, prompt_version="single-voter", batch=True)]
vote_preds = run(vote_chain, inputs, prompt_version="vote-of-three", batch=True)

expected = [e for _, e in EXAMPLES]
acc = lambda preds: sum(p == e for p, e in zip(preds, expected)) / len(expected)
log = tracker.load_log()
cost = lambda v: log[log.prompt_version == v].gwp_gCO2e.sum()
print(f"single voter: accuracy {acc(single_preds):.0%}, {cost('single-voter'):.4f} gCO2e")
print(f"vote of three: accuracy {acc(vote_preds):.0%}, {cost('vote-of-three'):.4f} gCO2e")
# The vote costs about three times more. It is worth it only if it fixes errors that matter;
# on easy examples it usually brings nothing.

## 5. Monolithic prompt or pipeline?

One large-model call that does everything, against our pipeline that uses the tiny model for extraction and verification and the small model for the brief. Same 10 texts.

In [ ]:
class MonolithicOutput(BaseModel):
    sdg: Optional[int] = Field(default=None, ge=1, le=17, description="Main SDG, null if none")
    brief: Optional[str] = Field(default=None, description="At most 80 words + 'Indicator to follow: ...', null if no SDG")
    grounded: bool = Field(description="True if every fact of the brief is stated in the text")


PROMPTS["mono-v1"] = ChatPromptTemplate.from_messages([
    ("system",
     "You analyse local news for an NGO tracking the UN SDGs (1-17). Identify the actors, the place and the "
     "action; decide the ONE SDG the text contributes to most directly (null if none); if there is one, write "
     "a brief of at most 80 words followed by 'Indicator to follow: ...'; then check that every fact of the "
     "brief is stated in the text. Use only facts stated in the text."),
    ("human", "<text>\n{text}\n</text>"),
])
mono_chain = (PROMPTS["mono-v1"] | large.with_structured_output(MonolithicOutput)).with_config(tags=["step:mono"])

inputs = [{"text": t} for t in TEXTS]
mono_results = run(mono_chain, inputs, prompt_version="mono-v1", batch=True)
pipe_results = run(pipeline, inputs, prompt_version="pipeline-v1-all", batch=True)

expected = [e for _, e in EXAMPLES]
mono_acc = sum(r.sdg == e for r, e in zip(mono_results, expected)) / len(expected)
pipe_acc = sum(r["analysis"].sdg == e for r, e in zip(pipe_results, expected)) / len(expected)

log = tracker.load_log()
comparison = (log[log.prompt_version.isin(["mono-v1", "pipeline-v1-all"])]
              .groupby("prompt_version")[["prompt_tokens", "completion_tokens", "latency_s", "gwp_gCO2e"]].sum())
comparison["sdg_accuracy"] = comparison.index.map({"mono-v1": mono_acc, "pipeline-v1-all": pipe_acc})
comparison

In [ ]:
# ✏️ Exercise 5 (solution)
for mono, pipe in list(zip(mono_results, pipe_results))[:3]:
    print("MONO:", mono.brief)
    print("PIPE:", pipe.get("brief", pipe.get("stopped")))
    print("-" * 60)
# a) If the accuracy is similar, the pipeline wins: smaller models, and no brief is written when the
#    gate stops (text 9).
# b) The estimated footprint depends mostly on the model size and on the output tokens: several calls
#    to a 3B or 24B model can weigh less than one call to a large model.
# c) Style, usefulness of the indicator, hallucinated details: quality aspects that accuracy on the
#    SDG number does not capture.

## 6. Bonus: evaluator-optimizer

A generator writes the brief, an evaluator checks it against explicit criteria and returns feedback, the generator revises. The number of rounds is **bounded**.

In [ ]:
class Review(BaseModel):
    under_80_words: bool
    has_measurable_indicator: bool
    only_facts_from_text: bool
    feedback: str = Field(description="What to change, in at most 2 sentences. Empty if everything is fine")


PROMPTS["review-v1"] = ChatPromptTemplate.from_messages([
    ("system", "You review an NGO brief against three criteria: at most 80 words; ends with a measurable "
               "indicator; uses only facts stated in the source text."),
    ("human", "<text>\n{text}\n</text>\n<brief>\n{brief}\n</brief>"),
])
PROMPTS["revise-v1"] = ChatPromptTemplate.from_messages([
    ("system", "Revise the brief following the feedback. Keep at most 80 words and the final indicator line. "
               "Use only facts stated in the text."),
    ("human", "<text>\n{text}\n</text>\n<brief>\n{brief}\n</brief>\n<feedback>\n{feedback}\n</feedback>"),
])
review_chain = (PROMPTS["review-v1"] | tiny.with_structured_output(Review)).with_config(tags=["step:review"])
revise_chain = (PROMPTS["revise-v1"] | small | StrOutputParser()).with_config(tags=["step:revise"])


def evaluator_optimizer(text, max_rounds=2):
    analysis = run(extract_chain, {"text": text}, prompt_version="eval-opt-v1")
    if analysis.sdg is None:
        return None, []
    state = {"text": text, "analysis": analysis}
    brief = run(BRIEF_CHAINS[FAMILY_OF[analysis.sdg]], state, prompt_version="eval-opt-v1")
    history = []
    for _ in range(max_rounds):
        review = run(review_chain, {"text": text, "brief": brief}, prompt_version="eval-opt-v1")
        history.append(review)
        if review.under_80_words and review.has_measurable_indicator and review.only_facts_from_text:
            break
        brief = run(revise_chain, {"text": text, "brief": brief, "feedback": review.feedback},
                    prompt_version="eval-opt-v1")
    return brief, history


brief, history = evaluator_optimizer(TEXTS[5])
print(brief)
for review in history:
    print(review)

In [ ]:
# ✏️ Exercise 6 (solution)
log = tracker.load_log()
per_step = log[log.prompt_version == "eval-opt-v1"].groupby("tag")["gwp_gCO2e"].agg(["count", "sum"])
print(per_step)
# The review and revise steps add one or two calls per round. Worth it for outputs read by people and
# hard to fix later (a public brief); not for intermediate outputs that code checks anyway.

## Wrap-up

* A pipeline splits a task into **simpler steps**, each with the **right model**.
* **Structured outputs** (Pydantic) make each step's result usable by code; values are still validated.
* **Gates and routers are plain Python**: test them without calling any model.
* `RunnableParallel` reduces latency, **not** tokens; votes and review loops multiply calls.
* Measure **per step**: the `TokenTracker` callback logs every call with its step tag.

**For your project, before leaving today**: replace `tracker.py` with version 2, and build a chain of at least two LLM calls with a structured output that runs on 5 examples of your use case (progress indicator of today). Post your update on Teams.

**Next session**: tools, the agent loop, the harness, MCP and skills.